# Final DataFlash Pipeline

Этот ноутбук фиксирует полный воспроизводимый путь до текущей итоговой `DataFlash` модели и связанных прогонов траекторий.

Что он делает:

- явно проверяет, какие сырые `DataFlash` логи реально есть в репозитории;
- пересобирает текущий best `DataFlash` pipeline одной командой;
- пересобирает comparison page по `GPS/POS`, `IMU`, `flow`, `POLI_NA` и best `DataFlash` rollout;
- показывает, где лежат финальные markdown- и HTML-артефакты.

Важно: на текущем состоянии репозитория `DataFlash` validation остается `single-log only`, потому что найден только один исходный `DataFlash` лог.

In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
ROOT = None
for candidate in candidates:
    if (candidate / 'src').exists() and (candidate / 'reports').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError('Project root not found')

os.chdir(ROOT)
ROOT

## 1. Проверка исходных DataFlash логов

Перед любыми выводами ноутбук явно показывает, есть ли независимые `DataFlash` полеты для межполетной проверки. Если лог только один, это ограничение должно остаться в итоговом отчете.

In [ ]:
raw_logs = sorted(ROOT.glob('artifacts/*.log'))
raw_bins = sorted(ROOT.glob('artifacts/*.bin'))
print('DataFlash .log files:')
for path in raw_logs:
    print(' -', path.relative_to(ROOT))
print('\nDataFlash .bin files:')
for path in raw_bins:
    print(' -', path.relative_to(ROOT))

print(f'\nIndependent DataFlash logs found: {len(raw_logs)}')
if len(raw_logs) <= 1:
    print('Status: single-log DataFlash validation only')
else:
    print('Status: multi-flight DataFlash validation is possible')

## 2. Пересборка best DataFlash pipeline

Эта команда делает полный воспроизводимый rebuild текущего best `DataFlash` кандидата: sequence baseline, prediction viewer, rollout reports, финальный markdown и финальный HTML dashboard.

In [ ]:
subprocess.run(['python3', 'src/run_best_dataflash_pipeline.py'], check=True)

## 3. Пересборка comparison page

Comparison page собирает в одну страницу доступные сценарии:

- `linear`: real GPS vs `flow` vs best available `POLI_NA` preset;
- `triangle`: real GPS vs `flow` vs best available `POLI_NA` preset;
- `dataflash`: real POS/GPS vs pure IMU DR vs best `DataFlash` rollout.

In [ ]:
subprocess.run(['python3', 'src/build_navigation_comparison.py'], check=True)

## 4. Ключевые итоговые отчеты

Здесь выводим основные versioned markdown-артефакты, на которые теперь должен опираться финальный рассказ о проекте.

In [ ]:
report_paths = [
    ROOT / 'reports' / 'final' / 'final_dataflash_report.md',
    ROOT / 'reports' / 'final' / 'final_report.md',
    ROOT / 'reports' / 'navigation' / 'navigation_comparison.md',
]

for path in report_paths:
    print(f'===== {path.relative_to(ROOT)} =====')
    text = path.read_text(encoding='utf-8')
    print('\n'.join(text.splitlines()[:40]))
    print()

## 5. Пути к итоговым артефактам

Этот блок нужен как короткая шпаргалка: где лежат финальные markdown- и HTML-результаты после rebuild.

In [ ]:
artifacts = [
    'reports/final/final_dataflash_report.md',
    'reports/final/final_report.md',
    'reports/navigation/navigation_comparison.md',
    'reports/experiments/dataflash_rollout_summary.md',
    'artifacts/generated/dataflash/final_report/index.html',
    'artifacts/generated/dataflash/predictions/sequence_fixed100_shrink/index.html',
    'artifacts/generated/dataflash/rollouts/sequence_fixed100_shrink/index.html',
    'artifacts/generated/navigation/comparison/index.html',
]

for rel in artifacts:
    path = ROOT / rel
    print(f'{rel} ->', 'OK' if path.exists() else 'MISSING')

## 6. Вывод

На текущем состоянии проекта корректный вывод такой:

- лучший практический результат в репозитории дает `DataFlash` модель `sequence_ridge_bias_tuned`;
- pure IMU без внешних ограничений непригоден;
- `flow` помогает, но нестабилен;
- `POLI_NA` ограничен отсутствием исходного preprocessing spec;
- честной межполетной `DataFlash` проверки пока нет, потому что в репозитории только один исходный `DataFlash` лог.

Следующий исследовательский шаг после этого ноутбука: добавить независимые `DataFlash` полеты и перевести validation со `single-log rolling folds` на `flight-level split`.